# Marketing Campaign Performance Analytics
## Notebook 01: Data Cleaning, Auditing & Feature Engineering

### Business Context
A digital marketing agency spends substantial budgets across multiple advertising channels (Google Ads, YouTube, Facebook, Instagram, Email, Website) but lacks a unified, data-driven understanding of campaign returns and customer behavior. 

The analytical objective of this portfolio project is to evaluate:
- **Return on Investment (ROI)** and **Return on Ad Spend (ROAS)**
- **Customer Acquisition Cost (CAC)** and **Cost Per Click (CPC)**
- **Click-Through Rate (CTR)** and **Conversion Rates**
- **Channel profitability** and **Customer segment economics**

> **Architecture Notice:** This project uses two raw datasets in `data/raw/`:
> 1. `marketing_campaign_dataset.csv` (Campaign performance metrics, ~200k rows)
> 2. `marketing_campaign.csv` (Customer demographic & personality profiles, ~2.2k rows)
>
> **These two datasets have NO shared key.** They represent two distinct analytical perspectives (macro channel performance vs. micro customer personality profiles). They must remain distinct tables and are never joined.

### 0. Environment Setup and Project Paths
We configure standard data manipulation packages (`pandas`, `numpy`, `matplotlib`, `seaborn`) and establish robust relative paths using Python's `pathlib.Path` so this notebook runs portably from any directory.

In [1]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Relative directory paths
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_DIR = BASE_DIR / 'data' / 'raw'
CLEAN_DATA_DIR = BASE_DIR / 'data' / 'cleaned'
REPORTS_DIR = BASE_DIR / 'reports'

# Ensure output directories exist
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base Project Directory : {BASE_DIR.resolve()}")
print(f"Raw Data Directory     : {RAW_DATA_DIR.resolve()}")
print(f"Cleaned Data Directory : {CLEAN_DATA_DIR.resolve()}")
print(f"Reports Directory      : {REPORTS_DIR.resolve()}")

Base Project Directory : C:\projects\marketing-campaign-analytics
Raw Data Directory     : C:\projects\marketing-campaign-analytics\data\raw
Cleaned Data Directory : C:\projects\marketing-campaign-analytics\data\cleaned
Reports Directory      : C:\projects\marketing-campaign-analytics\reports


---
## Part A: Data Inspection (Pre-Cleaning Audit)

**Goal:** Before mutating any raw data, inspect each dataset to understand its genuine structure, data types, distributions, anomalies, missing fields, and duplicates.
We do not guess schema properties — we let the data speak directly.

### A.1 Inspecting Dataset 1: Campaign Performance Data (`marketing_campaign_dataset.csv`)
We load the file and print:
- Table dimensions (`shape`) and column data types (`dtypes`)
- Sample records (`head`)
- Missing value counts per column
- Duplicate rows and duplicate `Campaign_ID` count
- Descriptive statistics (`describe`) for numeric features
- Frequency counts for all categorical features

In [2]:
campaign_raw_path = RAW_DATA_DIR / 'marketing_campaign_dataset.csv'
print(f"Reading raw campaign dataset from: {campaign_raw_path.name}...")
df_camp_raw = pd.read_csv(campaign_raw_path)

print(f"\n1. Dataset Dimensions: {df_camp_raw.shape[0]:,} rows x {df_camp_raw.shape[1]} columns")

print("\n2. Data Types:")
print(df_camp_raw.dtypes)

print("\n3. First 5 Rows:")
display(df_camp_raw.head())

print("\n4. Missing Values Per Column:")
null_camp = df_camp_raw.isnull().sum()
print(null_camp if null_camp.sum() > 0 else "Zero missing values detected across all columns.")

print(f"\n5. Duplicate Entire Rows: {df_camp_raw.duplicated().sum()}")
print(f"   Duplicate Campaign_IDs: {df_camp_raw['Campaign_ID'].duplicated().sum()}")

print("\n6. Summary Statistics (Numeric Columns):")
display(df_camp_raw.describe())

print("\n7. Categorical Columns Value Counts:")
camp_cat_cols = df_camp_raw.select_dtypes(include=['object', 'string']).columns
for col in camp_cat_cols:
    print(f"\n--- Value Counts for '{col}' (Top 10) ---")
    print(df_camp_raw[col].value_counts().head(10))

Reading raw campaign dataset from: marketing_campaign_dataset.csv...

1. Dataset Dimensions: 200,000 rows x 16 columns

2. Data Types:
Campaign_ID           int64
Company                 str
Campaign_Type           str
Target_Audience         str
Duration                str
Channel_Used            str
Conversion_Rate     float64
Acquisition_Cost        str
ROI                 float64
Location                str
Language                str
Clicks                int64
Impressions           int64
Engagement_Score      int64
Customer_Segment        str
Date                    str
dtype: object

3. First 5 Rows:


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,"$16,174.00",6.29,Chicago,Spanish,506,1922,6,Health & Wellness,01-01-21
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,"$11,566.00",5.61,New York,German,116,7523,7,Fashionistas,02-01-21
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,"$10,200.00",7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,03-01-21
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,"$12,724.00",5.55,Miami,Mandarin,217,1820,7,Health & Wellness,04-01-21
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,"$16,452.00",6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,05-01-21



4. Missing Values Per Column:
Zero missing values detected across all columns.

5. Duplicate Entire Rows: 0
   Duplicate Campaign_IDs: 0

6. Summary Statistics (Numeric Columns):


,Campaign_ID,Conversion_Rate,ROI,Clicks,Impressions,Engagement_Score
count,"200,000.00","200,000.00","200,000.00","200,000.00","200,000.00","200,000.00"
mean,"100,000.50",0.08,5.00,549.77,"5,507.30",5.49
std,"57,735.17",0.04,1.73,260.02,"2,596.86",2.87
min,1.00,0.01,2.00,100.00,"1,000.00",1.00
25%,"50,000.75",0.05,3.50,325.00,"3,266.00",3.00
50%,"100,000.50",0.08,5.01,550.00,"5,517.50",5.00
75%,"150,000.25",0.12,6.51,775.00,"7,753.00",8.00
max,"200,000.00",0.15,8.00,"1,000.00","10,000.00",10.00



7. Categorical Columns Value Counts:

--- Value Counts for 'Company' (Top 10) ---
Company
TechCorp               40237
Alpha Innovations      40051
DataTech Solutions     40012
NexGen Systems         39991
Innovate Industries    39709
Name: count, dtype: int64

--- Value Counts for 'Campaign_Type' (Top 10) ---
Campaign_Type
Influencer      40169
Search          40157
Display         39987
Email           39870
Social Media    39817
Name: count, dtype: int64

--- Value Counts for 'Target_Audience' (Top 10) ---
Target_Audience
Men 18-24      40258
Men 25-34      40023
All Ages       40019
Women 25-34    40013
Women 35-44    39687
Name: count, dtype: int64

--- Value Counts for 'Duration' (Top 10) ---
Duration
30 days    50255
45 days    50100
60 days    49866
15 days    49779
Name: count, dtype: int64

--- Value Counts for 'Channel_Used' (Top 10) ---
Channel_Used
Email         33599
Google Ads    33438
YouTube       33392
Instagram     33392
Website       33360
Facebook      32819
Name:

### A.2 Inspecting Dataset 2: Customer Personality Data (`marketing_campaign.csv`)
Notice that upon examining the raw text of `marketing_campaign.csv`, the file is **semicolon-separated (`;`)** and carries a UTF-8 Byte Order Mark (BOM). We read it using `sep=';'` and `encoding='utf-8-sig'` to parse all 29 columns accurately.

In [4]:
customer_raw_path = RAW_DATA_DIR / 'marketing_campaign.csv'
print(f"Reading raw customer dataset from: {customer_raw_path.name}...")
df_cust_raw = pd.read_csv(customer_raw_path, sep=';', encoding='utf-8-sig')

print(f"\n1. Dataset Dimensions: {df_cust_raw.shape[0]:,} rows x {df_cust_raw.shape[1]} columns")

print("\n2. Data Types:")
print(df_cust_raw.dtypes)

print("\n3. First 5 Rows:")
display(df_cust_raw.head())

print("\n4. Missing Values Per Column:")
null_cust = df_cust_raw.isnull().sum()
print(null_cust[null_cust > 0])

print(f"\n5. Duplicate Entire Rows: {df_cust_raw.duplicated().sum()}")
print(f"   Duplicate Customer IDs: {df_cust_raw['ID'].duplicated().sum()}")

print("\n6. Summary Statistics (Numeric Columns):")
display(df_cust_raw.describe())

print("\n7. Categorical Columns Value Counts:")
cust_cat_cols = df_cust_raw.select_dtypes(include=['object', 'string']).columns
for col in cust_cat_cols:
    print(f"\n--- Value Counts for '{col}' ---")
    print(df_cust_raw[col].value_counts(dropna=False))

Reading raw customer dataset from: marketing_campaign.csv...

1. Dataset Dimensions: 2,240 rows x 29 columns

2. Data Types:
ID                       int64
Year_Birth               int64
Education                  str
Marital_Status             str
Income                 float64
Kidhome                  int64
Teenhome                 int64
Dt_Customer                str
Recency                  int64
MntWines                 int64
MntFruits                int64
MntMeatProducts          int64
MntFishProducts          int64
MntSweetProducts         int64
MntGoldProds             int64
NumDealsPurchases        int64
NumWebPurchases          int64
NumCatalogPurchases      int64
NumStorePurchases        int64
NumWebVisitsMonth        int64
AcceptedCmp3             int64
AcceptedCmp4             int64
AcceptedCmp5             int64
AcceptedCmp1             int64
AcceptedCmp2             int64
Complain                 int64
Z_CostContact            int64
Z_Revenue                int64
Respons

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,"58,138.00",0,0,2012-09-04,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,"46,344.00",1,1,2014-03-08,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,"71,613.00",0,0,2013-08-21,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,"26,646.00",1,0,2014-02-10,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,"58,293.00",1,0,2014-01-19,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0



4. Missing Values Per Column:
Income    24
dtype: int64

5. Duplicate Entire Rows: 0
   Duplicate Customer IDs: 0

6. Summary Statistics (Numeric Columns):


,ID,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
count,"2,240.00","2,240.00","2,216.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00","2,240.00"
mean,"5,592.16","1,968.81","52,247.25",0.44,0.51,49.11,303.94,26.30,166.95,37.53,27.06,44.02,2.33,4.08,2.66,5.79,5.32,0.07,0.07,0.07,0.06,0.01,0.01,3.00,11.00,0.15
std,"3,246.66",11.98,"25,173.08",0.54,0.54,28.96,336.60,39.77,225.72,54.63,41.28,52.17,1.93,2.78,2.92,3.25,2.43,0.26,0.26,0.26,0.25,0.11,0.10,0.00,0.00,0.36
min,0.00,"1,893.00","1,730.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,3.00,11.00,0.00
25%,"2,828.25","1,959.00","35,303.00",0.00,0.00,24.00,23.75,1.00,16.00,3.00,1.00,9.00,1.00,2.00,0.00,3.00,3.00,0.00,0.00,0.00,0.00,0.00,0.00,3.00,11.00,0.00
50%,"5,458.50","1,970.00","51,381.50",0.00,0.00,49.00,173.50,8.00,67.00,12.00,8.00,24.00,2.00,4.00,2.00,5.00,6.00,0.00,0.00,0.00,0.00,0.00,0.00,3.00,11.00,0.00
75%,"8,427.75","1,977.00","68,522.00",1.00,1.00,74.00,504.25,33.00,232.00,50.00,33.00,56.00,3.00,6.00,4.00,8.00,7.00,0.00,0.00,0.00,0.00,0.00,0.00,3.00,11.00,0.00
max,"11,191.00","1,996.00","666,666.00",2.00,2.00,99.00,"1,493.00",199.00,"1,725.00",259.00,263.00,362.00,15.00,27.00,28.00,13.00,20.00,1.00,1.00,1.00,1.00,1.00,1.00,3.00,11.00,1.00



7. Categorical Columns Value Counts:

--- Value Counts for 'Education' ---
Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64

--- Value Counts for 'Marital_Status' ---
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64

--- Value Counts for 'Dt_Customer' ---
Dt_Customer
2012-08-31    12
2014-05-12    11
2012-09-12    11
2013-02-14    11
2014-05-22    10
              ..
2013-11-01     1
2013-08-10     1
2012-10-11     1
2012-12-20     1
2014-01-09     1
Name: count, Length: 663, dtype: int64


### A.3 Markdown Summary: Raw Data Quality Findings

#### Campaign Performance Dataset (`marketing_campaign_dataset.csv` - 200,000 records)
1. **`Acquisition_Cost` format issue:** Stored as string containing dollar signs (`$`), thousand separators (`,`), and trailing whitespace (e.g., `'$16,174.00 '`). Needs stripping and parsing to `float64`.
2. **`Duration` format issue:** Stored as string text with unit suffix (e.g., `'30 days'`). Needs regex extraction to integer days.
3. **`Date` format issue:** Stored as string in `DD-MM-YY` format (`01-01-21` through `31-12-21`). Must be parsed into standard datetime timestamps with day-first ordering.
4. **`Conversion_Rate` verification:** Values range from `0.01` to `0.15` (mean ~0.08). This represents a **decimal fraction** (1% to 15%), not a whole percentage number.
5. **Impossible values validation:**
   - `Clicks` range from 100 to 1,000; `Impressions` range from 1,000 to 10,000. `Clicks <= Impressions` holds true across 100% of rows.
   - `Acquisition_Cost` and `ROI` are strictly positive (zero negative cost rows).
6. **Missing / Duplicates:** Zero missing values and zero duplicate rows.

#### Customer Personality Dataset (`marketing_campaign.csv` - 2,240 records)
1. **File format:** Semicolon delimiter (`;`) with UTF-8 BOM encoding (`utf-8-sig`).
2. **Missing data:** `Income` is missing in exactly **24 rows** (~1.07%). All other 28 columns have zero nulls.
3. **Biological age outliers:** `Year_Birth` contains records dating back to `1893`, `1899`, and `1900`. For a dataset collected around 2014, this represents ages exceeding 114–121 years. These 3 rows are clear data entry errors.
4. **Extreme income outlier:** The maximum income is `$666,666`, whereas the 75th percentile is `$68,522` and the second highest is `$162,397`. This single anomalous entry will heavily skew statistical averages and regression models.
5. **Categorical inconsistencies:**
   - `Marital_Status`: Non-standard values include `'Together'` (580), `'Alone'` (3), `'Absurd'` (2), and `'YOLO'` (2).
   - `Education`: European university terminology (`'Graduation'` = Bachelor's degree, `'2n Cycle'` = Master's level).
6. **Zero-variance columns:** `Z_CostContact` (always 3) and `Z_Revenue` (always 11) have standard deviation 0.

---
## Part B: Clean the Campaign Dataset

In this section, we apply best-practice transformations without mutating slices or raw files:
1. **Strip currency symbols and convert `Acquisition_Cost` to float.**
2. **Convert `Duration` to integer days.**
3. **Parse `Date` as datetime with `%d-%m-%y`.**
4. **Verify and cast `Conversion_Rate` as float decimal fraction.**
5. **Standardize categorical text (strip whitespace, ensure proper title casing).**
6. **Audit and flag impossible values (cost $\le 0$, clicks > impressions, negative metrics).**
7. **Engineer derived metrics:**
   - $\text{Revenue} = \text{Acquisition Cost} \times (1 + \text{ROI})$
   - $\text{Profit} = \text{Revenue} - \text{Acquisition Cost}$
   - $\text{CTR} = \frac{\text{Clicks}}{\text{Impressions}}$ (guarded against division by zero)
   - $\text{CPC} = \frac{\text{Acquisition Cost}}{\text{Clicks}}$ (guarded against division by zero)
   - $\text{ROAS} = \frac{\text{Revenue}}{\text{Acquisition Cost}}$ (guarded against division by zero)
   - Calendar attributes: `Year`, `Month`, `Quarter`

In [5]:
# Make explicit working copy to avoid SettingWithCopyWarning
df_camp_clean = df_camp_raw.copy()

# 1. Clean Acquisition_Cost
df_camp_clean['Acquisition_Cost'] = (
    df_camp_clean['Acquisition_Cost']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(float)
)

# 2. Convert Duration to integer number of days
df_camp_clean['Duration'] = (
    df_camp_clean['Duration']
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
    .astype(int)
)

# 3. Parse Date to datetime
df_camp_clean['Date'] = pd.to_datetime(df_camp_clean['Date'], format='%d-%m-%y')

# 4. Confirm Conversion_Rate is a consistent decimal fraction
df_camp_clean['Conversion_Rate'] = df_camp_clean['Conversion_Rate'].astype(float)

# 5. Standardize categorical text fields
camp_text_fields = [
    'Company', 'Campaign_Type', 'Target_Audience', 
    'Channel_Used', 'Location', 'Language', 'Customer_Segment'
]
for col in camp_text_fields:
    df_camp_clean[col] = df_camp_clean[col].astype(str).str.strip().str.title()

# 6. Audit impossible values
impossible_cost = (df_camp_clean['Acquisition_Cost'] <= 0).sum()
impossible_clicks_imp = (df_camp_clean['Clicks'] > df_camp_clean['Impressions']).sum()
impossible_clicks_neg = (df_camp_clean['Clicks'] < 0).sum()
impossible_imp_neg = (df_camp_clean['Impressions'] <= 0).sum()
impossible_roi = (df_camp_clean['ROI'] < 0).sum()

print("=== Campaign Impossible Value Audit ===")
print(f"Acquisition_Cost <= $0 : {impossible_cost} rows affected")
print(f"Clicks > Impressions   : {impossible_clicks_imp} rows affected")
print(f"Negative Clicks        : {impossible_clicks_neg} rows affected")
print(f"Impressions <= 0       : {impossible_imp_neg} rows affected")
print(f"Negative ROI           : {impossible_roi} rows affected")

# 7. Derived Features (with divide-by-zero guards)
# Revenue = Acquisition_Cost * (1 + ROI)
df_camp_clean['Revenue'] = df_camp_clean['Acquisition_Cost'] * (1.0 + df_camp_clean['ROI'])

# Profit = Revenue - Acquisition_Cost
df_camp_clean['Profit'] = df_camp_clean['Revenue'] - df_camp_clean['Acquisition_Cost']

# CTR = Clicks / Impressions (guarded)
df_camp_clean['CTR'] = np.where(
    df_camp_clean['Impressions'] > 0, 
    df_camp_clean['Clicks'] / df_camp_clean['Impressions'], 
    0.0
)

# CPC = Acquisition_Cost / Clicks (guarded)
df_camp_clean['CPC'] = np.where(
    df_camp_clean['Clicks'] > 0, 
    df_camp_clean['Acquisition_Cost'] / df_camp_clean['Clicks'], 
    0.0
)

# ROAS = Revenue / Acquisition_Cost (guarded)
df_camp_clean['ROAS'] = np.where(
    df_camp_clean['Acquisition_Cost'] > 0, 
    df_camp_clean['Revenue'] / df_camp_clean['Acquisition_Cost'], 
    0.0
)

# Calendar Features
df_camp_clean['Year'] = df_camp_clean['Date'].dt.year
df_camp_clean['Month'] = df_camp_clean['Date'].dt.month
df_camp_clean['Quarter'] = df_camp_clean['Date'].dt.quarter

print(f"\nCleaned Campaign Dataset Dimensions: {df_camp_clean.shape[0]:,} rows x {df_camp_clean.shape[1]} columns")
display(df_camp_clean[['Campaign_ID', 'Company', 'Channel_Used', 'Acquisition_Cost', 'Revenue', 'Profit', 'CTR', 'CPC', 'ROAS', 'Date']].head())

=== Campaign Impossible Value Audit ===
Acquisition_Cost <= $0 : 0 rows affected
Clicks > Impressions   : 0 rows affected
Negative Clicks        : 0 rows affected
Impressions <= 0       : 0 rows affected
Negative ROI           : 0 rows affected

Cleaned Campaign Dataset Dimensions: 200,000 rows x 24 columns


,Campaign_ID,Company,Channel_Used,Acquisition_Cost,Revenue,Profit,CTR,CPC,ROAS,Date
0,1,Innovate Industries,Google Ads,"16,174.00","117,908.46","101,734.46",0.26,31.96,7.29,2021-01-01
1,2,Nexgen Systems,Google Ads,"11,566.00","76,451.26","64,885.26",0.02,99.71,6.61,2021-01-02
2,3,Alpha Innovations,Youtube,"10,200.00","83,436.00","73,236.00",0.08,17.47,8.18,2021-01-03
3,4,Datatech Solutions,Youtube,"12,724.00","83,342.20","70,618.20",0.12,58.64,6.55,2021-01-04
4,5,Nexgen Systems,Youtube,"16,452.00","123,390.00","106,938.00",0.09,43.41,7.50,2021-01-05


---
## Part C: Clean the Customer Dataset

In this section, we clean and enrich the customer dataset `marketing_campaign.csv`:
1. **Impute Missing Income:** 24 missing values are imputed using the **median income grouped by education tier**, preserving the observed socio-economic distribution.
2. **Handle Extreme Outliers:**
   - Filter out birth years prior to 1920 (removes 3 records: birth years 1893, 1899, 1900, representing biologically impossible ages > 94 in the 2014 survey window).
   - Filter out `Income > $600,000` (removes 1 record with `$666,666`, which is a recognized erroneous outlier).
3. **Standardize Categories:**
   - `Marital_Status`: Map `'Together'` $\rightarrow$ `'Partner'`, `'Alone'` $\rightarrow$ `'Single'`, `'Widow'` $\rightarrow$ `'Widowed'`, `'Absurd'` / `'YOLO'` $\rightarrow$ `'Other'`.
   - `Education`: Map `'Graduation'` $\rightarrow$ `'Graduate'`, `'2n Cycle'` $\rightarrow$ `'Master'`.
4. **Derived Customer Features:**
   - `Age`: Calculated as `2014 - Year_Birth` (2014 represents the dataset survey collection benchmark).
   - `Total_Spend`: Sum of all 6 product category expenditures (`MntWines`, `MntFruits`, `MntMeatProducts`, `MntFishProducts`, `MntSweetProducts`, `MntGoldProds`).
   - `Campaigns_Accepted`: Sum of binary response flags across marketing campaigns 1 through 5, plus the latest promotional campaign (`Response`).
   - `Age_Group`: Demographic cohorts (`'<30'`, `'30-44'`, `'45-59'`, `'60+'`).
   - `Income_Group`: Income tiers (`'Low (<$35k)'`, `'Middle ($35k-$65k)'`, `'High ($65k-$100k)'`, `'Affluent (>$100k)'`).

In [7]:
df_cust_clean = df_cust_raw.copy()

# 1. Impute missing Income using median by Education
income_median_by_edu = df_cust_clean.groupby('Education')['Income'].transform('median')
df_cust_clean['Income'] = df_cust_clean['Income'].fillna(income_median_by_edu)

# 2. Filter biological age outliers and extreme income outlier
initial_count = len(df_cust_clean)
valid_cust_mask = (df_cust_clean['Year_Birth'] >= 1920) & (df_cust_clean['Income'] <= 600000)
df_cust_clean = df_cust_clean[valid_cust_mask].copy()
dropped_count = initial_count - len(df_cust_clean)
print(f"Removed {dropped_count} outlier records (3 implausible birth years < 1920, 1 extreme income of $666,666).")

# 3. Standardize Marital_Status
marital_map = {
    'Married': 'Married',
    'Together': 'Partner',
    'Single': 'Single',
    'Divorced': 'Divorced',
    'Widow': 'Widowed',
    'Alone': 'Single',
    'Absurd': 'Other',
    'YOLO': 'Other'
}
df_cust_clean['Marital_Status'] = df_cust_clean['Marital_Status'].map(marital_map).fillna('Other')

# Standardize Education
edu_map = {
    'Basic': 'Basic',
    '2n Cycle': 'Master',
    'Graduation': 'Graduate',
    'Master': 'Master',
    'PhD': 'PhD'
}
df_cust_clean['Education'] = df_cust_clean['Education'].map(edu_map).fillna(df_cust_clean['Education'])

# Parse Dt_Customer to datetime
df_cust_clean['Dt_Customer'] = pd.to_datetime(df_cust_clean['Dt_Customer'], format='%Y-%m-%d')

# 4. Derived Columns
# Reference year 2014 reflects the survey collection window (2012 - mid 2014)
df_cust_clean['Age'] = 2014 - df_cust_clean['Year_Birth']

# Total product spend
spend_fields = [
    'MntWines', 'MntFruits', 'MntMeatProducts', 
    'MntFishProducts', 'MntSweetProducts', 'MntGoldProds'
]
df_cust_clean['Total_Spend'] = df_cust_clean[spend_fields].sum(axis=1)

# Total campaigns accepted
cmp_flags = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response']
df_cust_clean['Campaigns_Accepted'] = df_cust_clean[cmp_flags].sum(axis=1)

# Demographic Age Groups
age_bins = [0, 29, 44, 59, 120]
age_labels = ['<30', '30-44', '45-59', '60+']
df_cust_clean['Age_Group'] = pd.cut(df_cust_clean['Age'], bins=age_bins, labels=age_labels, right=True)

# Socioeconomic Income Groups
inc_bins = [0, 35000, 65000, 100000, np.inf]
inc_labels = ['Low (<$35k)', 'Middle ($35k-$65k)', 'High ($65k-$100k)', 'Affluent (>$100k)']
df_cust_clean['Income_Group'] = pd.cut(df_cust_clean['Income'], bins=inc_bins, labels=inc_labels, right=False)

print(f"Cleaned Customer Dataset Dimensions: {df_cust_clean.shape[0]:,} rows x {df_cust_clean.shape[1]} columns")
display(df_cust_clean[['ID', 'Age', 'Age_Group', 'Education', 'Marital_Status', 'Income', 'Income_Group', 'Total_Spend', 'Campaigns_Accepted']].head())

Removed 4 outlier records (3 implausible birth years < 1920, 1 extreme income of $666,666).
Cleaned Customer Dataset Dimensions: 2,236 rows x 34 columns


,ID,Age,Age_Group,Education,Marital_Status,Income,Income_Group,Total_Spend,Campaigns_Accepted
0,5524,57,45-59,Graduate,Single,"58,138.00",Middle ($35k-$65k),1617,1
1,2174,60,60+,Graduate,Single,"46,344.00",Middle ($35k-$65k),27,0
2,4141,49,45-59,Graduate,Partner,"71,613.00",High ($65k-$100k),776,0
3,6182,30,30-44,Graduate,Partner,"26,646.00",Low (<$35k),53,0
4,5324,33,30-44,PhD,Married,"58,293.00",Middle ($35k-$65k),422,0


---
## Part D: Cleaning Log, Exports, and Pipeline Validation

In this final stage, we:
1. Compile an audit log documenting all detected issues and remediation actions.
2. Export `reports/cleaning_log.csv`.
3. Export cleaned datasets to `data/cleaned/campaigns_clean.csv` and `data/cleaned/customers_clean.csv`.
4. Execute automated assertions to ensure production-grade data integrity.

In [8]:
# 1. Construct the Cleaning Audit Log
cleaning_log_entries = [
    {
        'dataset': 'marketing_campaign_dataset.csv',
        'issue': 'Acquisition_Cost stored as currency string with $ and commas',
        'rows_affected': len(df_camp_raw),
        'action_taken': 'Stripped $, commas, and whitespace; converted to float64'
    },
    {
        'dataset': 'marketing_campaign_dataset.csv',
        'issue': 'Duration stored as string ("X days")',
        'rows_affected': len(df_camp_raw),
        'action_taken': 'Extracted integer day counts using regex; converted to int64'
    },
    {
        'dataset': 'marketing_campaign_dataset.csv',
        'issue': 'Date stored as DD-MM-YY string',
        'rows_affected': len(df_camp_raw),
        'action_taken': 'Parsed with format %d-%m-%y into standard datetime64[ns]'
    },
    {
        'dataset': 'marketing_campaign_dataset.csv',
        'issue': 'Conversion_Rate scale consistency check',
        'rows_affected': 0,
        'action_taken': 'Verified values are valid decimal fractions (range 0.01 - 0.15)'
    },
    {
        'dataset': 'marketing_campaign_dataset.csv',
        'issue': 'Categorical text whitespace and casing',
        'rows_affected': len(df_camp_raw),
        'action_taken': 'Stripped leading/trailing whitespace and standardized title casing across 7 categorical columns'
    },
    {
        'dataset': 'marketing_campaign_dataset.csv',
        'issue': 'Impossible values validation (cost <= 0, clicks > impressions)',
        'rows_affected': 0,
        'action_taken': 'Audited for negative costs, impressions <= 0, and clicks > impressions; 0 anomalies detected'
    },
    {
        'dataset': 'marketing_campaign_dataset.csv',
        'issue': 'Feature engineering (financial & calendar KPIs)',
        'rows_affected': len(df_camp_raw),
        'action_taken': 'Derived Revenue, Profit, CTR, CPC, ROAS (with zero-division guards), Year, Month, Quarter'
    },
    {
        'dataset': 'marketing_campaign.csv',
        'issue': 'File format with semicolon delimiter and UTF-8 BOM',
        'rows_affected': len(df_cust_raw),
        'action_taken': 'Loaded using sep=";" and encoding="utf-8-sig"'
    },
    {
        'dataset': 'marketing_campaign.csv',
        'issue': 'Missing Income values',
        'rows_affected': 24,
        'action_taken': 'Imputed nulls using median income grouped by Education level'
    },
    {
        'dataset': 'marketing_campaign.csv',
        'issue': 'Impossible birth years (< 1920: 1893, 1899, 1900)',
        'rows_affected': 3,
        'action_taken': 'Removed biological outlier records (implied ages > 94 in 2014 survey collection)'
    },
    {
        'dataset': 'marketing_campaign.csv',
        'issue': 'Extreme income outlier ($666,666)',
        'rows_affected': 1,
        'action_taken': 'Removed outlier record exceeding reasonable economic threshold'
    },
    {
        'dataset': 'marketing_campaign.csv',
        'issue': 'Non-standard Marital_Status categories',
        'rows_affected': len(df_cust_raw),
        'action_taken': 'Standardized (Together -> Partner, Alone -> Single, Absurd/YOLO -> Other, Widow -> Widowed)'
    },
    {
        'dataset': 'marketing_campaign.csv',
        'issue': 'Non-standard Education categories',
        'rows_affected': len(df_cust_raw),
        'action_taken': 'Standardized higher education labels (Graduation -> Graduate, 2n Cycle -> Master)'
    },
    {
        'dataset': 'marketing_campaign.csv',
        'issue': 'Customer behavioral & demographic feature engineering',
        'rows_affected': len(df_cust_clean),
        'action_taken': 'Derived Age (ref 2014), Total_Spend, Campaigns_Accepted, Age_Group, Income_Group'
    }
]

df_cleaning_log = pd.DataFrame(cleaning_log_entries)
display(df_cleaning_log)

# Save Cleaning Log
cleaning_log_path = REPORTS_DIR / 'cleaning_log.csv'
df_cleaning_log.to_csv(cleaning_log_path, index=False)
print(f"\nSaved Cleaning Log to : {cleaning_log_path.resolve()}")

# Save Cleaned Datasets
camp_clean_path = CLEAN_DATA_DIR / 'campaigns_clean.csv'
cust_clean_path = CLEAN_DATA_DIR / 'customers_clean.csv'

df_camp_clean.to_csv(camp_clean_path, index=False)
df_cust_clean.to_csv(cust_clean_path, index=False)

print(f"Saved Cleaned Campaigns: {camp_clean_path.resolve()}")
print(f"Saved Cleaned Customers: {cust_clean_path.resolve()}")

,dataset,issue,rows_affected,action_taken
0,marketing_campaign_dataset.csv,Acquisition_Cost stored as currency string wit...,200000,"Stripped $, commas, and whitespace; converted ..."
1,marketing_campaign_dataset.csv,"Duration stored as string (""X days"")",200000,Extracted integer day counts using regex; conv...
2,marketing_campaign_dataset.csv,Date stored as DD-MM-YY string,200000,Parsed with format %d-%m-%y into standard date...
3,marketing_campaign_dataset.csv,Conversion_Rate scale consistency check,0,Verified values are valid decimal fractions (r...
4,marketing_campaign_dataset.csv,Categorical text whitespace and casing,200000,Stripped leading/trailing whitespace and stand...
5,marketing_campaign_dataset.csv,"Impossible values validation (cost <= 0, click...",0,"Audited for negative costs, impressions <= 0, ..."
6,marketing_campaign_dataset.csv,Feature engineering (financial & calendar KPIs),200000,"Derived Revenue, Profit, CTR, CPC, ROAS (with ..."
7,marketing_campaign.csv,File format with semicolon delimiter and UTF-8...,2240,"Loaded using sep="";"" and encoding=""utf-8-sig"""
8,marketing_campaign.csv,Missing Income values,24,Imputed nulls using median income grouped by E...
9,marketing_campaign.csv,"Impossible birth years (< 1920: 1893, 1899, 1900)",3,Removed biological outlier records (implied ag...



Saved Cleaning Log to : C:\projects\marketing-campaign-analytics\reports\cleaning_log.csv
Saved Cleaned Campaigns: C:\projects\marketing-campaign-analytics\data\cleaned\campaigns_clean.csv
Saved Cleaned Customers: C:\projects\marketing-campaign-analytics\data\cleaned\customers_clean.csv


### D.2 Pipeline Validation Block
Automated assertion suite checking column completeness, data types, valid ranges, and before/after row count deltas.

In [9]:
# --- Campaign Dataset Assertions ---
print("Running Campaign Data Validations...")
assert df_camp_clean['Acquisition_Cost'].isnull().sum() == 0, "Null values found in Acquisition_Cost!"
assert df_camp_clean['Revenue'].isnull().sum() == 0, "Null values found in Revenue!"
assert df_camp_clean['Profit'].isnull().sum() == 0, "Null values found in Profit!"
assert df_camp_clean['CTR'].isnull().sum() == 0, "Null values found in CTR!"
assert df_camp_clean['CPC'].isnull().sum() == 0, "Null values found in CPC!"
assert df_camp_clean['ROAS'].isnull().sum() == 0, "Null values found in ROAS!"
assert (df_camp_clean['Clicks'] <= df_camp_clean['Impressions']).all(), "Found Clicks > Impressions!"
assert (df_camp_clean['Acquisition_Cost'] > 0).all(), "Found non-positive Acquisition_Cost!"
assert pd.api.types.is_float_dtype(df_camp_clean['Acquisition_Cost']), "Acquisition_Cost is not float!"
assert pd.api.types.is_datetime64_any_dtype(df_camp_clean['Date']), "Date is not datetime64!"
assert pd.api.types.is_integer_dtype(df_camp_clean['Duration']), "Duration is not integer!"
print("✔ All Campaign assertions PASSED.")

# --- Customer Dataset Assertions ---
print("\nRunning Customer Data Validations...")
assert df_cust_clean['Income'].isnull().sum() == 0, "Null values found in Income!"
assert df_cust_clean['Age'].isnull().sum() == 0, "Null values found in Age!"
assert df_cust_clean['Total_Spend'].isnull().sum() == 0, "Null values found in Total_Spend!"
assert df_cust_clean['Campaigns_Accepted'].isnull().sum() == 0, "Null values found in Campaigns_Accepted!"
assert df_cust_clean['Age'].min() >= 18, f"Implausible minor age found: {df_cust_clean['Age'].min()}"
assert df_cust_clean['Age'].max() <= 100, f"Implausible senior age found: {df_cust_clean['Age'].max()}"
assert df_cust_clean['Income'].max() < 600000, f"Extreme income outlier not removed: {df_cust_clean['Income'].max()}"
assert df_cust_clean['Marital_Status'].isin(['Married', 'Partner', 'Single', 'Divorced', 'Widowed', 'Other']).all(), "Unstandardized marital status found!"
assert df_cust_clean['Education'].isin(['Basic', 'Graduate', 'Master', 'PhD']).all(), "Unstandardized education found!"
print("✔ All Customer assertions PASSED.")

# --- Row and Column Reconciliation ---
print("\n=== Pipeline Execution Reconciliation ===")
print(f"Campaign Rows     : {len(df_camp_raw):,} raw -> {len(df_camp_clean):,} clean (100.0% retention)")
print(f"Customer Rows     : {len(df_cust_raw):,} raw -> {len(df_cust_clean):,} clean ({len(df_cust_clean)/len(df_cust_raw):.2%} retention; 4 outlier rows dropped)")
print(f"Campaign Features : {df_camp_raw.shape[1]} raw -> {df_camp_clean.shape[1]} clean (+8 derived features)")
print(f"Customer Features : {df_cust_raw.shape[1]} raw -> {df_cust_clean.shape[1]} clean (+5 derived features)")

Running Campaign Data Validations...
✔ All Campaign assertions PASSED.

Running Customer Data Validations...
✔ All Customer assertions PASSED.

=== Pipeline Execution Reconciliation ===
Campaign Rows     : 200,000 raw -> 200,000 clean (100.0% retention)
Customer Rows     : 2,240 raw -> 2,236 clean (99.82% retention; 4 outlier rows dropped)
Campaign Features : 16 raw -> 24 clean (+8 derived features)
Customer Features : 29 raw -> 34 clean (+5 derived features)


In [11]:
camp_clean = pd.read_csv("../data/cleaned/campaigns_clean.csv")
cust_clean = pd.read_csv("../data/cleaned/customers_clean.csv")
print(camp_clean.dtypes)
print(cust_clean.dtypes)

Campaign_ID           int64
Company                 str
Campaign_Type           str
Target_Audience         str
Duration              int64
Channel_Used            str
Conversion_Rate     float64
Acquisition_Cost    float64
ROI                 float64
Location                str
Language                str
Clicks                int64
Impressions           int64
Engagement_Score      int64
Customer_Segment        str
Date                    str
Revenue             float64
Profit              float64
CTR                 float64
CPC                 float64
ROAS                float64
Year                  int64
Month                 int64
Quarter               int64
dtype: object
ID                       int64
Year_Birth               int64
Education                  str
Marital_Status             str
Income                 float64
Kidhome                  int64
Teenhome                 int64
Dt_Customer                str
Recency                  int64
MntWines                 int64
MntF

---
## Summary of Biggest Data Quality Issues Found

1. **Unformatted Currency and String Durations:** 100% of rows in `marketing_campaign_dataset.csv` stored `Acquisition_Cost` as uncast strings with currency symbols and trailing whitespace (`'$16,174.00 '`) and `Duration` with `' days'` units.
2. **Non-Standard CSV Delimiter and UTF-8 BOM:** `marketing_campaign.csv` used semicolon (`;`) separators with a Byte Order Mark rather than standard comma/tab formats, requiring explicit encoding and parsing handling.
3. **Missing Demographic Income:** `Income` contained 24 missing values (~1.07%), successfully imputed using education-cohort medians to preserve economic variance.
4. **Implausible Biological Ages:** 3 customer records had birth years from 1893 to 1900 (ages > 114–121 years), filtered as clear data entry errors.
5. **Severe Economic Outlier:** A single customer record reported `$666,666` in annual income (over 4x the 99th percentile), filtered out to prevent statistical distortion in downstream marketing analyses.

In [15]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
engine = create_engine(os.environ["DATABASE_URL"])

with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())

PostgreSQL 18.7 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


In [17]:
camp_clean = pd.read_csv("../data/cleaned/campaigns_clean.csv", parse_dates=["Date"])
camp_clean.columns = camp_clean.columns.str.lower()
camp_clean = camp_clean.rename(columns={"duration": "duration_days", "date": "campaign_date"})

camp_clean.to_sql("campaigns", engine, if_exists="append", index=False,
                  chunksize=5000, method="multi")

200000

In [18]:
print(len(camp_clean))
print(pd.read_sql("SELECT COUNT(*) FROM campaigns", engine))

200000
    count
0  200000


In [19]:
cust_clean = pd.read_csv("../data/cleaned/customers_clean.csv")
cust_clean.columns = cust_clean.columns.str.lower()
cust_clean.to_sql("customers", engine, if_exists="replace", index=False)

print(len(cust_clean))
print(pd.read_sql("SELECT COUNT(*) FROM customers", engine))

2236
   count
0   2236
